<a href="https://colab.research.google.com/github/ekBlaise/huggingface_solution_measuring_time/blob/master/LoRA_Explanation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer

In [ ]:
# MODEL_NAME = "google/gemma-2-9b-it"
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

| Parameter                          | Simple Meaning                        | Usually Change?      |
| ---------------------------------- | ------------------------------------- | -------------------- |
| `load_in_8bit`                     | Load model in INT8                    | ✅ Yes                |
| `load_in_4bit`                     | Load model in 4-bit                   | ✅ Yes                |
| `llm_int8_threshold`               | Outlier cutoff for FP16 vs INT8       | ❌ No                 |
| `llm_int8_skip_modules`            | Layers to keep unquantized            | Sometimes            |
| `llm_int8_enable_fp32_cpu_offload` | Move some layers to CPU               | Sometimes            |
| `llm_int8_has_fp16_weight`         | Keep FP16 copy for training           | Training only        |
| `bnb_4bit_compute_dtype`           | Precision used for computation        | ✅ Often (`bfloat16`) |
| `bnb_4bit_quant_type`              | Choose FP4 or NF4                     | ✅ Usually `nf4`      |
| `bnb_4bit_use_double_quant`        | Quantize scaling constants too        | ✅ Often for QLoRA    |
| `bnb_4bit_quant_storage`           | How 4-bit values are packed in memory | ❌ Rarely             |


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

| Parameter           | What it controls               | Recommended                                                         |
| ------------------- | ------------------------------ | ------------------------------------------------------------------- |
| `r`                 | Adapter size (capacity)        | 8–32                                                                |
| `target_modules`    | Which layers get LoRA          | `q_proj`, `k_proj`, `v_proj`, `o_proj` (often also MLP projections) |
| `lora_alpha`        | Strength of the LoRA update    | `2 × r` is a common starting point                                  |
| `lora_dropout`      | Regularization during training | `0.05`                                                              |
| `bias`              | Whether to train bias terms    | `"none"`                                                            |
| `modules_to_save`   | Extra layers to save           | Usually `None`                                                      |
| `use_rslora`        | More stable scaling            | `True` for many modern setups                                       |
| `init_lora_weights` | Adapter initialization         | `True` (default)                                                    |


In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj","up_proj","down_proj"
],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
lora_adapter_model = get_peft_model(base_model, lora_config)

In [ ]:
lora_adapter_model.print_trainable_parameters()

In [ ]:
total_params = lora_adapter_model.num_parameters()
trainable_params = lora_adapter_model.num_parameters(only_trainable=True)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Trainable %:", 100 * trainable_params / total_params)

In [ ]:
print(lora_adapter_model.peft_config)